# SCC0276 - Aprendizado de Máquina - EXERCÍCIO 4

1. Arthur Henrique Silva de Araujo - 14651458
2. Leonardo Doro Demore - 15674786


# Carregando e preparando dados para o exercício

No exercício desta semana, o objetivo é comparar três algoritmos de classificação utilizando um conjunto de dados de [digitos escritos a mão](https://archive.ics.uci.edu/dataset/80/optical+recognition+of+handwritten+digits). Neste conjunto de dados, cacda digito é representado por uma matriz (imagem) 8x8. No entanto, iremos tratar cada posição da matriz como um atributo ao invés de classificar a imagem como um todo.  

A seguir, os dados serão carregados e processados de forma a criar conjuntos de treino e teste.

In [28]:
from sklearn.datasets import load_digits

In [29]:
# carregando os dados
raw = load_digits(as_frame=True)

In [30]:
print(raw.DESCR)

.. _digits_dataset:

Optical recognition of handwritten digits dataset
--------------------------------------------------

**Data Set Characteristics:**

:Number of Instances: 1797
:Number of Attributes: 64
:Attribute Information: 8x8 image of integer pixels in the range 0..16.
:Missing Attribute Values: None
:Creator: E. Alpaydin (alpaydin '@' boun.edu.tr)
:Date: July; 1998

This is a copy of the test set of the UCI ML hand-written digits datasets
https://archive.ics.uci.edu/ml/datasets/Optical+Recognition+of+Handwritten+Digits

The data set contains images of hand-written digits: 10 classes where
each class refers to a digit.

Preprocessing programs made available by NIST were used to extract
normalized bitmaps of handwritten digits from a preprinted form. From a
total of 43 people, 30 contributed to the training set and different 13
to the test set. 32x32 bitmaps are divided into nonoverlapping blocks of
4x4 and the number of on pixels are counted in each block. This generates
an in

In [32]:
# separando atributos e classes
X = raw.data.values
y = raw.target

In [33]:
raw.data.info()

<class 'pandas.DataFrame'>
RangeIndex: 1797 entries, 0 to 1796
Data columns (total 64 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   pixel_0_0  1797 non-null   float64
 1   pixel_0_1  1797 non-null   float64
 2   pixel_0_2  1797 non-null   float64
 3   pixel_0_3  1797 non-null   float64
 4   pixel_0_4  1797 non-null   float64
 5   pixel_0_5  1797 non-null   float64
 6   pixel_0_6  1797 non-null   float64
 7   pixel_0_7  1797 non-null   float64
 8   pixel_1_0  1797 non-null   float64
 9   pixel_1_1  1797 non-null   float64
 10  pixel_1_2  1797 non-null   float64
 11  pixel_1_3  1797 non-null   float64
 12  pixel_1_4  1797 non-null   float64
 13  pixel_1_5  1797 non-null   float64
 14  pixel_1_6  1797 non-null   float64
 15  pixel_1_7  1797 non-null   float64
 16  pixel_2_0  1797 non-null   float64
 17  pixel_2_1  1797 non-null   float64
 18  pixel_2_2  1797 non-null   float64
 19  pixel_2_3  1797 non-null   float64
 20  pixel_2_4  1797 non

In [34]:
from sklearn.model_selection import train_test_split

In [35]:
# separando os dados em treino e teste
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=2026, stratify=y)


# 1) Árvores de Decisão

Avalie [árvores de decisão](https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeClassifier.html) utilizando os conjuntos de treino e teste criados anteriormente e  cconfigurando a *seed* para o valor 2026. Reporte **acurácia**, **precisão** e **revocação (sensibilidade)** para os dados de teste. Para este exercício, **avalie três configurações** de parâmetros para o modelo. Configure a *seed* para o valor 2026.

In [36]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score
import pandas as pd

# Três configurações para a arvore de decisão
dt_configs = [
    {"criterion": "gini", "max_depth": 100, "min_samples_split": 2},
    {"criterion": "entropy", "max_depth": 10, "min_samples_split": 2},
    {"criterion": "gini", "max_depth": 5, "min_samples_split": 10},
]

dt_results = []

for i, params in enumerate(dt_configs, start=1):
    model = DecisionTreeClassifier(random_state=2026, **params)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    dt_results.append({
        "config": f"Config {i}",
        "params": params,
        "accuracy": accuracy_score(y_test, y_pred),
        "precision_macro": precision_score(y_test, y_pred, average="macro", zero_division=0),
        "recall_macro": recall_score(y_test, y_pred, average="macro", zero_division=0),
    })

dt_results_df = pd.DataFrame(dt_results)
dt_results_df

,config,params,accuracy,precision_macro,recall_macro
0,Config 1,"{'criterion': 'gini', 'max_depth': 100, 'min_s...",0.855556,0.855429,0.855495
1,Config 2,"{'criterion': 'entropy', 'max_depth': 10, 'min...",0.916667,0.919617,0.916632
2,Config 3,"{'criterion': 'gini', 'max_depth': 5, 'min_sam...",0.672222,0.744425,0.671688


# 2) Florestas Aleatórias

Repita o exercício anterior, considerando [Florestas Aleatórias](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html). Lembre-se de avaliar três variações de parâmetros.

In [37]:
from sklearn.ensemble import RandomForestClassifier

# Três configurações para o random forest
rf_configs = [
    {"n_estimators": 100, "max_depth": None, "max_features": "sqrt"},
    {"n_estimators": 300, "max_depth": 15, "max_features": "sqrt"},
    {"n_estimators": 200, "max_depth": 8, "max_features": 0.5},
]

rf_results = []

for i, params in enumerate(rf_configs, start=1):
    model = RandomForestClassifier(random_state=2026, n_jobs=-1, **params)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    rf_results.append({
        "config": f"Config {i}",
        "params": params,
        "accuracy": accuracy_score(y_test, y_pred),
        "precision_macro": precision_score(y_test, y_pred, average="macro", zero_division=0),
        "recall_macro": recall_score(y_test, y_pred, average="macro", zero_division=0),
    })

rf_results_df = pd.DataFrame(rf_results)
rf_results_df

,config,params,accuracy,precision_macro,recall_macro
0,Config 1,"{'n_estimators': 100, 'max_depth': None, 'max_...",0.975000,0.975550,0.974912
1,Config 2,"{'n_estimators': 300, 'max_depth': 15, 'max_fe...",0.980556,0.980881,0.980393
2,Config 3,"{'n_estimators': 200, 'max_depth': 8, 'max_fea...",0.955556,0.956018,0.955384


# 3) XGBoost

Novamente, repita o exercício 1, avaliando o modelo *ensembles* com *boosting* implementados pela biblioteca [xgboost](https://xgboost.readthedocs.io/en/release_3.2.0/parameter.html). Lembre-se de avaliar três variações de parâmetros para o modelo. 

In [38]:
from xgboost import XGBClassifier

In [40]:
# Três configurações para o xgboost
xgb_configs = [
    {"eta": 0.3, "max_depth": 6, "gamma": 0},
    {"eta": 0, "max_depth": 0, "gamma": 0},
    {"eta": 0.5, "max_depth": 4, "gamma": 0.2},
]

xgb_results = []

for i, params in enumerate(xgb_configs, start=1):
    model = XGBClassifier(random_state=2026, **params)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    xgb_results.append({
        "config": f"Config {i}",
        "params": params,
        "accuracy": accuracy_score(y_test, y_pred),
        "precision_macro": precision_score(y_test, y_pred, average="macro", zero_division=0),
        "recall_macro": recall_score(y_test, y_pred, average="macro", zero_division=0),
    })

xgb_results_df = pd.DataFrame(xgb_results)
xgb_results_df

,config,params,accuracy,precision_macro,recall_macro
0,Config 1,"{'eta': 0.3, 'max_depth': 6, 'gamma': 0}",0.966667,0.967031,0.966349
1,Config 2,"{'eta': 0, 'max_depth': 0, 'gamma': 0}",0.102778,0.010278,0.100000
2,Config 3,"{'eta': 0.5, 'max_depth': 4, 'gamma': 0.2}",0.975000,0.975116,0.974762


# 4) Ensemble de MLPs (opcional)

Como último exercício, o desafio é implementar *ensembles* usando MLPs. Confira a documentação dos *ensembles* baseados em votação e **bagging**, e construa um de cada utilizando MLPs como estimador. Treine e avalie os modelos nos conjuntos de treino e teste criados anteriormente, reportando as mesmas métricas escolhidas.

In [ ]:
from sklearn.ensemble import VotingClassifier, BaggingClassifier
from sklearn.neural_network import MLPClassifier

## Questões teóricas para estudo

Não é necessário responder as perguntas a seguir para completar a atividade prática. Utilize as perguntas para estudo posteriormente.

- O que é um método de *ensemble* em aprendizado de máquina? Uma floresta aleatória corresponde a qual tipo de *ensemble*?
- Explique como *bootstrapping* afeta o funcionamento de árvores aleatórias.
- Qual a diferença entre *boosting* e *bagging*?
- Explique a ideia central do *gradient boosting* e o diferencie do método de *ensemble* usado pelas florestas aleatórias.
- Qual o papel do "gradiente" dentro do contexto de *gradient boosting*?
